# Pass Tagging Tool — v2

### Ball colour key
| Colour | Meaning |
|---|---|
| 🟡 Yellow | Real tracker detection — correct unless you relocate |
| 🟠 Orange | Smoother-extrapolated — verify and relocate if wrong |
| 🟢 Green  | Your manual override |
| _(none)_  | Completely missing — must relocate before tagging |

### Workflow
1. Set `VIDEO_PATH` and `TRACKS_CSV` in **Configuration**.
2. Run all cells — the UI appears after the last one.
3. Navigate to each frame. If the ball marker is wrong or missing, press **🎯 Relocate ball** and click the ball on the image (works on any frame — yellow, orange or missing).
4. Navigate to the kick frame → **Tag Departure**.
5. Navigate to the receiver contact frame → **Tag Arrival**.
6. Set **Outcome** and optional notes → **✔ Confirm pass**.
7. **💾 Save** writes `pass_events.csv` (safe to save mid-session).
8. Run the **Merge** cell to produce the final `per_frame_tracks_with_passes.csv`.

**Pass timing:** departure = frame ball visibly leaves the foot; arrival = frame receiver makes first contact.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
VIDEO_PATH  = "/content/playbook/HILAL-HAZM_match_B_up8.mp4"
TRACKS_CSV  = "/content/playbook/output/per_frame_tracks.csv"
EVENTS_CSV  = "/content/playbook/output/pass_events.csv"
MERGED_CSV  = "/content/playbook/output/per_frame_tracks_with_passes.csv"

BALL_CLASS   = 0
GK_CLASS     = 1
PLAYER_CLASS = 2
REF_CLASS    = 3

CANVAS_W     = 960   # display width in pixels (height auto from video aspect)

In [ ]:
# ── Installs + imports ─────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ipyevents", "-q"])

import cv2, os
import numpy as np
import pandas as pd
import ipywidgets as widgets
from ipyevents import Event
from IPython.display import display as ipy_display
from io import BytesIO
from PIL import Image as PILImage, ImageDraw

print("Imports OK")

In [ ]:
# ── Load tracking data ─────────────────────────────────────────────────────
df_tracks = pd.read_csv(TRACKS_CSV)

df_ball = df_tracks[df_tracks["class_id"] == BALL_CLASS].copy()
df_ball["cx"] = (df_ball["x1"] + df_ball["x2"]) / 2
df_ball["cy"] = (df_ball["y1"] + df_ball["y2"]) / 2
ball_idx = df_ball.set_index("frame")

df_players = df_tracks[df_tracks["class_id"] != BALL_CLASS].copy()
df_players["cx"] = (df_players["x1"] + df_players["x2"]) / 2
df_players["cy"] = (df_players["y1"] + df_players["y2"]) / 2
players_grp = df_players.groupby("frame")

cap          = cv2.VideoCapture(VIDEO_PATH)
FPS          = cap.get(cv2.CAP_PROP_FPS) or 25.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
CANVAS_H     = int(CANVAS_W * VID_H / VID_W)
_SX          = VID_W / CANVAS_W   # canvas px → frame px
_SY          = VID_H / CANVAS_H
has_pitch    = all(c in df_tracks.columns for c in ["x_m", "y_m"])

print(f"Video   : {TOTAL_FRAMES} frames @ {FPS:.2f} fps  ({VID_W}×{VID_H})")
print(f"Canvas  : {CANVAS_W}×{CANVAS_H}")
print(f"Tracks  : {len(df_tracks)} rows | ball: {len(df_ball)} | players/ref/gk: {len(df_players)}")
print(f"Pitch coords: {has_pitch}")

In [ ]:
# ── Load or initialise events ──────────────────────────────────────────────
if os.path.exists(EVENTS_CSV):
    _df = pd.read_csv(EVENTS_CSV)
    if "notes" not in _df.columns:        # older files predate the notes column
        _df["notes"] = ""
    _df["notes"] = _df["notes"].fillna("")
    events  = _df.to_dict("records")
    next_id = int(_df["pass_id"].max()) + 1 if len(_df) else 1
    print(f"Loaded {len(events)} existing events from {EVENTS_CSV}")
else:
    events, next_id = [], 1
    print("Starting fresh — no existing events file.")

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────

ball_overrides: dict[int, tuple[float, float]] = {}  # frame → (cx, cy)

def read_frame(idx: int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, bgr = cap.read()
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if ok else None


def get_ball(idx: int) -> tuple:
    """Returns (cx, cy, source).  source: 'manual'|'tracker'|'interp'|'missing'."""
    if idx in ball_overrides:
        return (*ball_overrides[idx], "manual")
    if idx in ball_idx.index:
        row = ball_idx.loc[idx]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        src = "interp" if bool(row.get("ball_interpolated", False)) else "tracker"
        return float(row.cx), float(row.cy), src
    return None, None, "missing"


def get_pitch_xy(idx: int) -> tuple:
    if not has_pitch or idx not in ball_idx.index:
        return None, None
    row = ball_idx.loc[idx]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    try:
        return float(row["x_m"]), float(row["y_m"])
    except (TypeError, ValueError, KeyError):
        return None, None


def nearest_player(idx: int, cx: float, cy: float) -> tuple:
    """Returns (display_track_id, distance_px) of closest player."""
    if idx not in players_grp.groups:
        return None, float("inf")
    grp = players_grp.get_group(idx)
    dists = np.hypot(grp.cx - cx, grp.cy - cy)
    best = dists.idxmin()
    return int(grp.loc[best, "display_track_id"]), float(dists[best])


_TEAM_COL = {0: (255, 105, 180), 1: (0, 200, 255), -1: (200, 200, 200)}

def _pcol(row):
    if row.get("class_id") == REF_CLASS:
        return (255, 140, 0)
    tid = int(row.get("team_id", -1)) if pd.notna(row.get("team_id")) else -1
    return _TEAM_COL.get(tid, (200, 200, 200))


_BALL_COL = {"tracker": (255, 230, 0), "interp": (255, 140, 0),
             "manual": (0, 255, 120), "missing": (150, 150, 150)}
_BALL_TAG = {"tracker": "BALL", "interp": "BALL(interp)",
             "manual": "BALL(manual)", "missing": ""}


def annotate(frame, idx, dep_info=None, arr_info=None) -> bytes:
    img = PILImage.fromarray(frame)
    d   = ImageDraw.Draw(img)
    sc  = img.width / 1280

    # Players
    if idx in players_grp.groups:
        for _, row in players_grp.get_group(idx).iterrows():
            x1, y1, x2, y2 = int(row.x1), int(row.y1), int(row.x2), int(row.y2)
            col = _pcol(row)
            d.rectangle([x1, y1, x2, y2], outline=col, width=max(1, int(2*sc)))
            d.text((x1, max(0, y1-14)), f"P{int(row.display_track_id)}", fill=col)

    # Ball
    cx, cy, src = get_ball(idx)
    if cx is not None:
        r  = max(8, int(10*sc))
        lw = max(2, int(3*sc))
        col = _BALL_COL[src]
        d.ellipse([cx-r, cy-r, cx+r, cy+r], outline=col, width=lw)
        tag = _BALL_TAG[src]
        if tag:
            d.text((cx+r+3, cy-8), tag, fill=col)

    # Pending departure marker (green ring)
    if dep_info:
        dx, dy = dep_info["dep_ball_px"], dep_info["dep_ball_py"]
        r2 = max(14, int(18*sc))
        d.ellipse([dx-r2, dy-r2, dx+r2, dy+r2], outline=(0, 255, 80), width=3)
        d.text((dx+r2+3, dy-10), f"DEP  P{dep_info['passer_id']}", fill=(0, 255, 80))

    # Pending arrival marker (blue ring)
    if arr_info:
        ax, ay = arr_info["arr_ball_px"], arr_info["arr_ball_py"]
        r2 = max(14, int(18*sc))
        d.ellipse([ax-r2, ay-r2, ax+r2, ay+r2], outline=(80, 180, 255), width=3)
        d.text((ax+r2+3, ay-10), f"ARR  P{arr_info['receiver_id']}", fill=(80, 180, 255))

    # Frame stamp
    d.text((8, 8), f"Frame {idx}  {idx/FPS:.2f}s", fill=(255, 255, 255))

    buf = BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return buf.getvalue()


print("Helpers ready.")

In [ ]:
# ── Interactive UI ─────────────────────────────────────────────────────────

state = {"frame": 0, "pending": {}, "events": events, "next_id": next_id}

# Image display widget (replaces ipycanvas — works reliably in Colab/JupyterLab)
w_image = widgets.Image(
    format="jpeg",
    layout=widgets.Layout(
        width=f"{CANVAS_W}px",
        height=f"{CANVAS_H}px",
        border="3px solid #444",
    ),
)

def _draw(jpeg: bytes):
    img = PILImage.open(BytesIO(jpeg)).resize((CANVAS_W, CANVAS_H))
    buf = BytesIO()
    img.save(buf, format="jpeg", quality=85)
    w_image.value = buf.getvalue()

# Navigation
w_slider = widgets.IntSlider(min=0, max=TOTAL_FRAMES-1, step=1, value=0,
                              description="Frame:", continuous_update=False,
                              layout=widgets.Layout(width="95%"))
w_step   = widgets.BoundedIntText(value=1, min=1, max=200,
                                   description="Step:",
                                   layout=widgets.Layout(width="110px"))
w_b10    = widgets.Button(description="◀◀10", layout=widgets.Layout(width="60px"))
w_b5     = widgets.Button(description="◀5",   layout=widgets.Layout(width="55px"))
w_prev   = widgets.Button(description="◀", layout=widgets.Layout(width="50px"))
w_next   = widgets.Button(description="▶", layout=widgets.Layout(width="50px"))
w_f5     = widgets.Button(description="5▶",   layout=widgets.Layout(width="55px"))
w_f10    = widgets.Button(description="10▶▶", layout=widgets.Layout(width="60px"))
w_gf_val = widgets.BoundedIntText(value=0, min=0, max=TOTAL_FRAMES-1,
                                   description="Go:",
                                   layout=widgets.Layout(width="120px"))
w_gf_btn = widgets.Button(description="Go", layout=widgets.Layout(width="50px"))

# Relocate toggle
w_relocate = widgets.ToggleButton(
    value=False, description="🎯 Relocate ball", button_style="warning",
    tooltip="Activate, then click the ball on the image.",
    layout=widgets.Layout(width="160px"))
w_clear_ov = widgets.Button(description="Clear override",
                             layout=widgets.Layout(width="130px"))
w_rel_hint = widgets.HTML("")

# Tagging
w_dep    = widgets.Button(description="Tag Departure", button_style="success",
                           layout=widgets.Layout(width="140px"))
w_arr    = widgets.Button(description="Tag Arrival",   button_style="info",
                           layout=widgets.Layout(width="130px"))
w_cancel = widgets.Button(description="Cancel",        button_style="warning",
                           layout=widgets.Layout(width="80px"))
w_undo   = widgets.Button(description="Undo last",     button_style="danger",
                           layout=widgets.Layout(width="100px"))
w_save   = widgets.Button(description="💾 Save",        button_style="primary",
                           layout=widgets.Layout(width="90px"))

# Confirm row
w_outcome = widgets.Dropdown(
    options=[("Complete","complete"),("Incomplete","incomplete"),("Unknown","unknown")],
    value="complete", description="Outcome:",
    layout=widgets.Layout(width="185px"))
w_notes   = widgets.Text(description="Notes:", placeholder="optional",
                          layout=widgets.Layout(width="260px"))
w_confirm = widgets.Button(description="✔ Confirm pass", button_style="success",
                            layout=widgets.Layout(width="145px"))
confirm_row = widgets.HBox([w_outcome, w_notes, w_confirm])
confirm_row.layout.display = "none"

w_ball_info = widgets.HTML("")
w_status    = widgets.HTML("")
w_table     = widgets.Output()


# ── Refresh ────────────────────────────────────────────────────────────────
def refresh(idx=None):
    if idx is None:
        idx = state["frame"]
    rgb = read_frame(idx)
    if rgb is None:
        return
    p = state["pending"]
    _draw(annotate(rgb, idx,
                   dep_info=p if "dep_frame" in p else None,
                   arr_info=p if "arr_frame" in p else None))

    cx, cy, src = get_ball(idx)
    src_html = {
        "tracker": "<b style='color:#aaf'>tracker ✓</b>",
        "interp" : "<b style='color:orange'>interp — verify!</b>",
        "manual" : "<b style='color:#8f8'>manual override</b>",
        "missing": "<b style='color:red'>missing — relocate before tagging</b>",
    }[src]
    if cx is not None:
        xm, ym = get_pitch_xy(idx)
        pstr = f" &nbsp; pitch ({xm:.0f}, {ym:.0f}) cm" if xm else ""
        w_ball_info.value = f"Ball: {src_html} &nbsp; px ({cx:.0f}, {cy:.0f}){pstr}"
    else:
        w_ball_info.value = f"Ball: {src_html}"

    if not p:
        w_status.value = "<b>Status:</b> idle — navigate to kick frame → <b>Tag Departure</b>"
    elif "arr_frame" not in p:
        w_status.value = (f"<b>Status:</b> departure @ frame <b>{p['dep_frame']}</b> "
                          f"(P{p['passer_id']}) — navigate to contact frame → <b>Tag Arrival</b>")
    else:
        w_status.value = "<b>Status:</b> both frames tagged — set outcome and <b>✔ Confirm</b>"


def refresh_table():
    with w_table:
        w_table.clear_output(wait=True)
        if state["events"]:
            cols = ["pass_id","departure_frame","departure_time",
                    "passer_id","arrival_frame","arrival_time",
                    "receiver_id","duration_s","outcome"]
            ipy_display(pd.DataFrame(state["events"])[cols].tail(10))
        else:
            print("No passes tagged yet.")


# ── Navigation ─────────────────────────────────────────────────────────────
def _go(idx):
    idx = max(0, min(TOTAL_FRAMES-1, idx))
    state["frame"] = idx
    w_slider.value = idx
    refresh(idx)

w_slider.observe(lambda c: (_go(c["new"]),), names="value")
w_prev.on_click(lambda _: _go(state["frame"] - w_step.value))
w_next.on_click(lambda _: _go(state["frame"] + w_step.value))
w_b10.on_click(lambda _: _go(state["frame"] - 10))
w_b5.on_click(lambda _: _go(state["frame"] - 5))
w_f5.on_click(lambda _: _go(state["frame"] + 5))
w_f10.on_click(lambda _: _go(state["frame"] + 10))
w_gf_btn.on_click(lambda _: _go(w_gf_val.value))


# ── Relocate: click on image via ipyevents ─────────────────────────────────
# ipyevents captures real DOM click events — reliable in Colab and JupyterLab.
def _on_relocate_toggle(change):
    if change["new"]:
        w_image.layout.border = "3px solid orange"
        w_image.layout.cursor = "crosshair"
        w_rel_hint.value = "<span style='color:orange'>Click on the ball in the image</span>"
    else:
        w_image.layout.border = "3px solid #444"
        w_image.layout.cursor = "default"
        w_rel_hint.value = ""

w_relocate.observe(_on_relocate_toggle, names="value")

_img_events = Event(source=w_image, watched_events=["click"])

def _on_image_click(event):
    if not w_relocate.value:
        return
    # offsetX/Y: position within the element in CSS px (= canvas px here)
    x = float(event.get("offsetX", event.get("relativeX", 0)))
    y = float(event.get("offsetY", event.get("relativeY", 0)))
    idx = state["frame"]
    fx, fy = x * _SX, y * _SY   # canvas px → frame px
    ball_overrides[idx] = (fx, fy)
    w_relocate.value = False     # auto-exit crosshair mode after one click
    w_rel_hint.value = (f"<span style='color:#8f8'>✔ Override set: "
                        f"({fx:.0f}, {fy:.0f}) on frame {idx}</span>")
    refresh(idx)

_img_events.on_dom_event(_on_image_click)


def _on_clear_override(_):
    idx = state["frame"]
    ball_overrides.pop(idx, None)
    w_rel_hint.value = f"<span style='color:orange'>Override cleared for frame {idx}</span>"
    refresh(idx)

w_clear_ov.on_click(_on_clear_override)


# ── Tag callbacks ──────────────────────────────────────────────────────────
def _tag_dep(_):
    idx = state["frame"]
    cx, cy, src = get_ball(idx)
    if cx is None:
        w_status.value = "<b style='color:red'>No ball — press 🎯 and click the ball first.</b>"
        return
    pid, dist = nearest_player(idx, cx, cy)
    xm, ym = get_pitch_xy(idx)
    state["pending"] = {
        "dep_frame": idx,  "dep_time": round(idx/FPS, 3),
        "dep_ball_px": cx, "dep_ball_py": cy,
        "dep_ball_xm": xm, "dep_ball_ym": ym, "dep_src": src,
        "passer_id": pid,  "passer_dist_px": round(dist, 1),
    }
    confirm_row.layout.display = "none"
    refresh(idx)


def _tag_arr(_):
    if "dep_frame" not in state["pending"]:
        w_status.value = "<b style='color:red'>Tag Departure first.</b>"
        return
    idx = state["frame"]
    cx, cy, src = get_ball(idx)
    if cx is None:
        w_status.value = "<b style='color:red'>No ball — press 🎯 and click the ball first.</b>"
        return
    pid, dist = nearest_player(idx, cx, cy)
    xm, ym = get_pitch_xy(idx)
    state["pending"].update({
        "arr_frame": idx,  "arr_time": round(idx/FPS, 3),
        "arr_ball_px": cx, "arr_ball_py": cy,
        "arr_ball_xm": xm, "arr_ball_ym": ym, "arr_src": src,
        "receiver_id": pid, "receiver_dist_px": round(dist, 1),
    })
    confirm_row.layout.display = ""
    refresh(idx)


def _confirm(_):
    p = state["pending"]
    if "dep_frame" not in p or "arr_frame" not in p:
        return
    df = p["dep_frame"]; af = p["arr_frame"]
    state["events"].append({
        "pass_id":           state["next_id"],
        "departure_frame":   df, "departure_time":   p["dep_time"],
        "passer_id":         p["passer_id"],
        "passer_dist_px":    p["passer_dist_px"],
        "ball_dep_px":       round(p["dep_ball_px"], 1),
        "ball_dep_py":       round(p["dep_ball_py"], 1),
        "ball_dep_xm":       p["dep_ball_xm"],
        "ball_dep_ym":       p["dep_ball_ym"],
        "dep_src":           p["dep_src"],
        "arrival_frame":     af, "arrival_time":     p["arr_time"],
        "receiver_id":       p["receiver_id"],
        "receiver_dist_px":  p["receiver_dist_px"],
        "ball_arr_px":       round(p["arr_ball_px"], 1),
        "ball_arr_py":       round(p["arr_ball_py"], 1),
        "ball_arr_xm":       p["arr_ball_xm"],
        "ball_arr_ym":       p["arr_ball_ym"],
        "arr_src":           p["arr_src"],
        "duration_frames":   af - df,
        "duration_s":        round((af - df) / FPS, 3),
        "outcome":           w_outcome.value,
        "notes":             w_notes.value.strip(),
    })
    state["next_id"] += 1
    state["pending"] = {}
    confirm_row.layout.display = "none"
    w_notes.value = ""
    refresh_table()
    refresh()


def _cancel(_):
    state["pending"] = {}
    confirm_row.layout.display = "none"
    refresh()


def _undo(_):
    if state["events"]:
        state["next_id"] = state["events"][-1]["pass_id"]
        state["events"].pop()
        refresh_table()
        refresh()


def _save(_):
    os.makedirs(os.path.dirname(EVENTS_CSV), exist_ok=True)
    pd.DataFrame(state["events"]).to_csv(EVENTS_CSV, index=False)
    w_status.value = (f"<b style='color:#8f8'>✔ Saved {len(state['events'])} "
                      f"passes → {EVENTS_CSV}</b>")


w_dep.on_click(_tag_dep)
w_arr.on_click(_tag_arr)
w_confirm.on_click(_confirm)
w_cancel.on_click(_cancel)
w_undo.on_click(_undo)
w_save.on_click(_save)


# ── Layout ─────────────────────────────────────────────────────────────────
ui = widgets.VBox([
    w_image,
    w_slider,
    widgets.HBox([w_b10, w_b5, w_prev, w_next, w_f5, w_f10, w_step, w_gf_val, w_gf_btn]),
    w_ball_info,
    widgets.HBox([w_relocate, w_clear_ov, w_rel_hint]),
    widgets.HBox([w_dep, w_arr, w_cancel, w_undo, w_save]),
    confirm_row,
    w_status,
    widgets.HTML("<hr><b>Last 10 tagged passes:</b>"),
    w_table,
])

ipy_display(ui)
refresh_table()
refresh(0)

In [ ]:
# ── Merge: pass events + ball overrides + spline-filled trajectory ─────────
#
# New columns on the per-frame table:
#   pass_id / pass_role / pass_phase
#   departure_time / arrival_time / duration_s / outcome
#   ball_source  –  'tracker' | 'manual' | 'spline' | 'interp'
#
# Ball trajectory:
#   Anchors = real detections + your manual clicks (manual wins on same frame).
#   A cubic spline through the anchors fills every missing / extrapolated frame.
#   Gaps > MAX_SPLINE_GAP between anchors are left blank (no blind guessing).

import numpy as np
from scipy.interpolate import CubicSpline

SPLINE_FILL    = True
MAX_SPLINE_GAP = 30   # max frames between anchors to bridge


def _build_anchors(df_tracks, ball_overrides):
    anchors = {}
    for _, r in df_tracks[df_tracks["class_id"] == BALL_CLASS].iterrows():
        f = int(r["frame"])
        if f not in anchors and int(r.get("ball_interpolated", 0)) == 0:
            anchors[f] = ((r.x1+r.x2)/2, (r.y1+r.y2)/2)
    for f, pos in ball_overrides.items():
        anchors[int(f)] = (float(pos[0]), float(pos[1]))
    return dict(sorted(anchors.items()))


def _spline_fill(anchors, max_gap):
    if len(anchors) < 2:
        return {}
    fs = np.array(list(anchors), dtype=float)
    xs = np.array([anchors[int(f)][0] for f in fs])
    ys = np.array([anchors[int(f)][1] for f in fs])
    if len(fs) >= 4:
        fx, fy = CubicSpline(fs, xs), CubicSpline(fs, ys)
    else:
        fx = lambda q: np.interp(q, fs, xs)
        fy = lambda q: np.interp(q, fs, ys)
    out = {}
    fia = fs.astype(int)
    for a, b in zip(fia[:-1], fia[1:]):
        if 1 < (b-a) <= max_gap:
            for f in range(a+1, b):
                out[f] = (float(fx(f)), float(fy(f)))
    return out


def merge_full(df_tracks, events, ball_overrides):
    df = df_tracks.copy()

    # Remove duplicate ball rows
    dup = (df["class_id"]==BALL_CLASS) & df.duplicated(subset=["frame","class_id"], keep="first")
    df  = df[~dup].reset_index(drop=True)

    # ── Pass tags ──────────────────────────────────────────────────────────
    for col in ["pass_id","pass_role","pass_phase"]:
        df[col] = pd.NA
    for ev in events:
        pid, d_f, a_f = ev["pass_id"], ev["departure_frame"], ev["arrival_frame"]
        m = (df["class_id"]==BALL_CLASS) & (df["frame"]>=d_f) & (df["frame"]<=a_f)
        df.loc[m, "pass_id"]    = pid
        df.loc[m, "pass_phase"] = "in_flight"
        if ev.get("passer_id") is not None:
            m = (df["frame"]==d_f) & (df["display_track_id"]==ev["passer_id"])
            df.loc[m, ["pass_id","pass_role","pass_phase"]] = [pid,"passer","departure"]
        if ev.get("receiver_id") is not None:
            m = (df["frame"]==a_f) & (df["display_track_id"]==ev["receiver_id"])
            df.loc[m, ["pass_id","pass_role","pass_phase"]] = [pid,"receiver","arrival"]

    # ── Denormalize time / outcome ─────────────────────────────────────────
    if events:
        df = df.merge(
            pd.DataFrame(events)[["pass_id","departure_time","arrival_time",
                                   "duration_s","outcome","notes"]],
            on="pass_id", how="left")
    else:
        for c in ["departure_time","arrival_time","duration_s","outcome","notes"]:
            df[c] = pd.NA

    # ── ball_source label ──────────────────────────────────────────────────
    df["ball_source"] = pd.NA
    ib = df["class_id"] == BALL_CLASS
    df.loc[ib & (df["ball_interpolated"]==0), "ball_source"] = "tracker"
    df.loc[ib & (df["ball_interpolated"]==1), "ball_source"] = "interp"

    # ── Ball half-size (for synthesised rows) ──────────────────────────────
    df_b  = df[ib]
    half_w = float(((df_b.x2-df_b.x1)/2).median()) if len(df_b) else 8.0
    half_h = float(((df_b.y2-df_b.y1)/2).median()) if len(df_b) else 8.0

    def _upsert(f, cx, cy, source):
        m = (df["class_id"]==BALL_CLASS) & (df["frame"]==f)
        if m.any():
            df.loc[m, ["x1","y1","x2","y2"]] = [cx-half_w, cy-half_h, cx+half_w, cy+half_h]
            df.loc[m, ["x_m","y_m"]] = None
            df.loc[m, "ball_source"] = source
            return None
        ref = df[df["frame"]==f]
        return {
            "frame":f, "track_id":-1, "display_track_id":-1,
            "class_id":BALL_CLASS, "conf":1.0,
            "x1":cx-half_w, "y1":cy-half_h, "x2":cx+half_w, "y2":cy+half_h,
            "x_m":None, "y_m":None, "team_id":-1,
            "homography_ok": bool(ref["homography_ok"].iloc[0]) if len(ref) else False,
            "kp_used": int(ref["kp_used"].iloc[0]) if len(ref) else 0,
            "inlier_ratio":0.0, "reproj_err":0.0,
            "detector_ran":1, "homography_state":"manual",
            "ball_interpolated":0,
            "pass_id":pd.NA, "pass_role":pd.NA, "pass_phase":pd.NA,
            "departure_time":pd.NA, "arrival_time":pd.NA,
            "duration_s":pd.NA, "outcome":pd.NA, "notes":pd.NA,
            "ball_source":source,
        }

    # ── Apply manual overrides ─────────────────────────────────────────────
    new_rows = []
    for f, (cx, cy) in ball_overrides.items():
        r = _upsert(int(f), float(cx), float(cy), "manual")
        if r: new_rows.append(r)

    # ── Spline-fill gaps ───────────────────────────────────────────────────
    n_spline = 0
    if SPLINE_FILL:
        for f, (cx, cy) in _spline_fill(_build_anchors(df_tracks, ball_overrides),
                                         MAX_SPLINE_GAP).items():
            r = _upsert(f, cx, cy, "spline")
            if r: new_rows.append(r)
            n_spline += 1

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    # Re-tag spline-filled ball rows that fall inside a pass window
    for ev in events:
        m = (df["class_id"]==BALL_CLASS) & df["pass_id"].isna() & \
            (df["frame"]>=ev["departure_frame"]) & (df["frame"]<=ev["arrival_frame"])
        df.loc[m, "pass_id"]    = ev["pass_id"]
        df.loc[m, "pass_phase"] = "in_flight"

    print(f"Manual overrides : {len(ball_overrides)} frames")
    print(f"Spline-filled    : {n_spline} frames")
    return df.sort_values(["frame","class_id"]).reset_index(drop=True)


# ── Run & save ─────────────────────────────────────────────────────────────
df_merged = merge_full(df_tracks, state["events"], ball_overrides)
df_merged.to_csv(MERGED_CSV, index=False)

ib = df_merged["class_id"] == BALL_CLASS
print(f"\nSaved → {MERGED_CSV}")
print(f"Ball rows by source:")
ipy_display(df_merged.loc[ib, "ball_source"].value_counts())
tf = df_merged["frame"].nunique()
print(f"Ball coverage: {df_merged.loc[ib,'frame'].nunique()}/{tf} frames "
      f"({df_merged.loc[ib,'frame'].nunique()/tf:.1%})")

In [ ]:
# ── Quick stats ────────────────────────────────────────────────────────────
if state["events"]:
    df_ev = pd.DataFrame(state["events"])
    print(f"Passes tagged    : {len(df_ev)}")
    print(f"Complete         : {(df_ev.outcome=='complete').sum()}")
    print(f"Incomplete       : {(df_ev.outcome=='incomplete').sum()}")
    print(f"Completion rate  : {(df_ev.outcome=='complete').mean():.1%}")
    print(f"Avg duration     : {df_ev.duration_s.mean():.2f}s")
    print()
    print("Passes per passer:")
    ipy_display(df_ev.groupby("passer_id").size().rename("passes").sort_values(ascending=False))
    print("\nBall source at departure:")
    ipy_display(df_ev.groupby("dep_src").size())
else:
    print("No passes tagged yet.")